# CPU 精準視覺化對接 (Interactive Precision Docking)
專為「先導藥物優化 (Lead Optimization)」與「結合機制研究」設計。
強調100%的使用者掌控度、即時的3D視覺互動，以及最高精度的單一分子對接演算。

### 環境建置與核心套件載入
功能說明：一鍵安裝對接與視覺化所需的所有底層套件（包含 AutoDock Vina CPU 引擎、py3Dmol 3D 渲染引擎、RDKit 與 OpenBabel）。

核心設計：採用靜默安裝 (-q) 保持版面整潔，並確保所有化學資訊學套件的版本相容性，為後續的 3D 運算打下基礎。

In [ ]:
# 隱藏安裝輸出來保持版面整潔
!pip install -q meeko vina py3Dmol ipywidgets biopython rdkit gemmi prody
!apt-get install -q -y openbabel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.0/294.0 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 75.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.1/37.1 MB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.5/37.5 MB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.1/103.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 84.2 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  libboost-iostreams1.74.0 libinchi1 libmaeparser1 libopenbabel7
The following NEW packages will be installed:
  libboost-iostreams1.74.0 libinchi1 libmaeparser1 libopenbabel7 openbabel
0 upgraded, 5 newly installed, 0 to remove and 3 not

In [ ]:
# 系統環境與檔案讀寫 (System & I/O)
import os
import sys
import time
import zipfile
import glob
import subprocess
from google.colab import files  # Colab 專屬檔案上傳/下載

# 數值運算與資料解析 (Math & Data Parsing)
import math             # 用於處理比對問題
import re              # 用於解析 Vina 的文字報告
import numpy as np        # 用於計算蛋白質質心與網格邊界
from datetime import datetime  # 用於產生打包檔案的時間標記

# 互動介面與視覺化 (UI & Visualization)
import ipywidgets as widgets              # UI 滑桿、按鈕與下拉選單
import py3Dmol                       # 3D 蛋白質與小分子渲染
from IPython.display import clear_output, display   # 控制 Colab 畫面刷新與顯示

# 生物結構處理與對接引擎 (Bioinformatics & Docking)
from Bio.PDB import PDBParser              # 讀取與解析 PDB 結構檔案
from vina import Vina                   # AutoDock Vina 核心運算引擎
from rdkit import Chem                   # 化學處理工具

print("[系統] 模組載入成功。")

[系統] 模組載入成功。


### 受體與配體上傳及標準化前處理
功能說明：處理使用者上傳之受體 (.pdb/.cif) 與配體 (.sdf/.mol2/.pdb) 檔案，系統將自動進行拓樸清整、補氫、電荷計算，最後封裝為 Vina 標準的 .pdbqt 格式。

核心設計：導入「原子名稱 100% 映射對齊 (Atom-Name Mapping)」演算法，徹底解決傳統轉檔軟體（如 Meeko/OpenBabel）容易遺失原子編號與殘基名稱的缺陷，確保對接前後的化學 Metadata 具備絕對一致性。

In [ ]:
# 環境清理 (解決重複上傳問題)
cleanup_patterns = ["receptor*", "ligand*", "output*", "*.pdbqt", "*.mol2", "*.sdf", "temp_*"]
for pattern in cleanup_patterns:
    for file_path in glob.glob(pattern):
        try:
            os.remove(file_path)
        except OSError:
            pass
print("[系統] 舊有暫存檔案已清理完畢。")

print("\n[步驟 1/2] 上傳 Receptor (.pdb 或 .cif)")
uploaded_rec = files.upload()
if not uploaded_rec: raise Exception("未上傳受體檔案，請重新執行！")

raw_rec = list(uploaded_rec.keys())[0]
rec_ext = raw_rec.split('.')[-1].lower()
# ★ 標準化：不論使用者叫什麼，內部一律改為 receptor_original.副檔名
receptor_file = f"receptor_original.{rec_ext}"
os.rename(raw_rec, receptor_file)
print(f"[提示] 受體已重新命名為系統標準: {receptor_file}")

print("\n[步驟 2/2] 上傳 Ligand (.sdf, .mol2 或 .pdb)")
uploaded_lig = files.upload()
if not uploaded_lig: raise Exception("未上傳配體檔案，請重新執行！")

raw_lig = list(uploaded_lig.keys())[0]
lig_ext = raw_lig.split('.')[-1].lower()
# ★ 標準化：不論使用者叫什麼，內部一律改為 ligand_original.副檔名
ligand_file = f"ligand_original.{lig_ext}"
os.rename(raw_lig, ligand_file)
print(f"[提示] 配體已重新命名為系統標準: {ligand_file}")

temp_files = []

[系統] 舊有暫存檔案已清理完畢。

[步驟 1/2] 上傳 Receptor (.pdb 或 .cif)


Saving receptor.pdb to receptor.pdb
[提示] 受體已重新命名為系統標準: receptor_original.pdb

[步驟 2/2] 上傳 Ligand (.sdf, .mol2 或 .pdb)


Saving Conformer3D_COMPOUND_CID_5287836 (1).sdf to Conformer3D_COMPOUND_CID_5287836 (1).sdf
[提示] 配體已重新命名為系統標準: ligand_original.sdf


In [ ]:
print(f"\n[處理] 轉換受體: {receptor_file} -> receptor.pdbqt")
actual_receptor_file = receptor_file

# 如果上傳的是 cif，先用 obabel 轉成 pdb
if rec_ext == 'cif':
    subprocess.run(f'obabel -i cif "{receptor_file}" -o pdb -O temp_receptor.pdb', shell=True, check=True)
    actual_receptor_file = "temp_receptor.pdb"
    temp_files.append(actual_receptor_file)

# 呼叫 Meeko 工具將受體轉為 pdbqt
subprocess.run(
    f'mk_prepare_receptor.py -i "{actual_receptor_file}" -o receptor -p',
    shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)


print(f"\n[處理] 進行配體 Atom Name 專一化與殘基命名 (LIG)...")

def patch_pdbqt_from_pdb(pdbqt_path, pdb_path):
    """
    PDB 國際標準對齊法 (Cols 12-27 全區塊精準替換版)：
    嚴格遵循 PDB Fixed-width 欄位定義，將原子名稱、殘基名稱與編號等 metadata 完整轉移。
    """
    # 1. 讀取 PDB，建立「座標 -> Metadata 區塊」字典
    pdb_atoms = []
    with open(pdb_path, 'r') as f:
        for line in f:
            if line.startswith("ATOM  ") or line.startswith("HETATM"):
                try:
                    # 嚴格擷取座標 (Cols 31-38, 39-46, 47-54)
                    x = float(line[30:38])
                    y = float(line[38:46])
                    z = float(line[46:54])

                    # 擷取第 12~27 欄 (Python index 11~26)
                    # 包含：空欄、原子名稱、替換位置、殘基名稱、空欄、鏈代號、殘基編號、插入碼
                    metadata_block = line[11:27]
                    pdb_atoms.append({'x': x, 'y': y, 'z': z, 'block': metadata_block})
                except ValueError:
                    pass # 略過斷行或不符規範的異常行

    # 2. 讀取 PDBQT 並動手術
    with open(pdbqt_path, 'r') as f:
        pdbqt_lines = f.readlines()

    new_lines = []
    for line in pdbqt_lines:
        if line.startswith("ATOM  ") or line.startswith("HETATM"):
            try:
                # 嚴格擷取 PDBQT 座標
                x = float(line[30:38])
                y = float(line[38:46])
                z = float(line[46:54])

                min_dist = 999.0
                best_block = line[11:27] # 預設保留 PDBQT 原本的 12~27 欄

                # 尋找 3D 空間中最接近的 PDB 原子
                for patom in pdb_atoms:
                    dist = (x - patom['x'])**2 + (y - patom['y'])**2 + (z - patom['z'])**2
                    if dist < min_dist:
                        min_dist = dist
                        best_block = patom['block']

                # 將容忍度嚴格縮小至 0.01 (即實際空間距離 < 0.1 埃)
                if min_dist < 0.01:
                    # 完美拼接：保留行首 1~11 欄，替換 12~27 欄，保留 28 欄以後所有資料
                    line = line[:11] + best_block + line[27:]

            except ValueError:
                pass

        new_lines.append(line)

    # 3. 覆寫存回硬碟
    with open(pdbqt_path, 'w') as f:
        f.write("".join(new_lines))

def prepare_ligand_for_md(input_path, res_name="LIG"):
    """
    單一對接專用 (PDB 直連對齊版)
    """
    global temp_files
    if 'temp_files' not in globals():
        temp_files = []

    subprocess.run(f'obabel "{input_path}" -o sdf -O temp_conv.sdf', shell=True, stderr=subprocess.DEVNULL)
    temp_files.append('temp_conv.sdf')

    supplier = Chem.SDMolSupplier('temp_conv.sdf', removeHs=False)
    mol = supplier[0] if supplier else None

    if mol is None:
        raise ValueError("RDKit 解析配體失敗，請檢查原始結構。")

    element_counts = {}
    for atom in mol.GetAtoms():
        symbol = atom.GetSymbol()
        element_counts[symbol] = element_counts.get(symbol, 0) + 1
        atom_name = f"{symbol}{element_counts[symbol]}"

        mi = Chem.AtomPDBResidueInfo()
        mi.SetName(atom_name.ljust(4))
        mi.SetResidueName(res_name)
        mi.SetResidueNumber(1)
        mi.SetIsHeteroAtom(True)
        atom.SetMonomerInfo(mi)

    # 1. 產生完美 PDB
    pdb_out = "ligand.pdb"
    Chem.MolToPDBFile(mol, pdb_out)

    # 2. 轉出 MOL2
    mol2_out = "ligand.mol2"
    subprocess.run(f'obabel -i pdb "{pdb_out}" -o mol2 -O "{mol2_out}" --title "{res_name}"', shell=True, stderr=subprocess.DEVNULL)

    # 3. CLI 產生 PDBQT
    pdbqt_out = "ligand.pdbqt"
    subprocess.run(f'mk_prepare_ligand.py -i "{mol2_out}" -o "{pdbqt_out}"', shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    # 4. ★ PDB 直連對齊法：直接拿剛做好的 PDB 來修復 PDBQT
    if os.path.exists(pdb_out) and os.path.exists(pdbqt_out):
        patch_pdbqt_from_pdb(pdbqt_out, pdb_out)
    else:
        raise ValueError("Meeko CLI PDBQT 轉換失敗，找不到輸出檔案。")

    return mol2_out, pdb_out

# 執行配體處理主流程
try:
    final_mol2, final_pdb = prepare_ligand_for_md(ligand_file)
    print(f"       已生成專一化檔案: {final_mol2} & {final_pdb}")
    print(f"[處理] 轉換配體: {final_mol2} -> ligand.pdbqt (已完成原子與殘基欄位 100% 映射對齊)")
except Exception as e:
    print(f"[錯誤] 配體處理流程中斷: {e}")

# 3. 驗證與清理
print("\n[驗證] 最終檔案檢查...")
if os.path.exists("receptor.pdbqt") and os.path.exists("ligand.pdbqt"):
    print("[成功] 所有對接所需 PDBQT 檔案已就緒！")
else:
    print("[失敗] 轉檔發生錯誤，請檢查輸入檔案。")

# 徹底清理中間暫存檔
for tmp in temp_files:
    if os.path.exists(tmp): os.remove(tmp)


[處理] 轉換受體: receptor_original.pdb -> receptor.pdbqt

[處理] 進行配體 Atom Name 專一化與殘基命名 (LIG)...
       已生成專一化檔案: ligand.mol2 & ligand.pdb
[處理] 轉換配體: ligand.mol2 -> ligand.pdbqt (已完成原子與殘基欄位 100% 映射對齊)

[驗證] 最終檔案檢查...
[成功] 所有對接所需 PDBQT 檔案已就緒！


### 3D 互動式視角與 Grid Box 動態定位
功能說明：於 Colab 介面中直接渲染蛋白質三維結構，提供研究人員透過滑鼠拖曳、旋轉來精細觀察結合位點 (Binding Pocket)。

核心設計：建構「Grid Box 視覺化動態追蹤」機制。使用者輸入空間座標與尺寸後，畫面上的搜索網格會即時更新，確保對接區域在三維空間中完全包覆目標口袋，實現「所見即所得」的精準空間定位。

In [ ]:
parser = PDBParser(QUIET=True)
structure = parser.get_structure('receptor', actual_receptor_file)

heavy_atoms = [atom.coord for atom in structure.get_atoms() if atom.element != 'H']
coords = np.array(heavy_atoms)

centroid = np.mean(coords, axis=0)
initial_center = {k: round(v, 1) for k, v in zip(['cx', 'cy', 'cz'], centroid)}

spans = np.max(coords, axis=0) - np.min(coords, axis=0)
padding = (np.max(spans) / 2) + 30

print(f"[資訊] 蛋白質跨度: {np.max(spans):.2f} Å | 動態邊界鎖定: 質心 +/- {padding:.2f} Å")

vina_box = {**initial_center, 'sx': 20, 'sy': 20, 'sz': 20}

with open(actual_receptor_file, 'r') as f:
    pdb_data = f.read()

view = py3Dmol.view(width=800, height=500)
view.addModel(pdb_data, 'pdb')
view.setStyle({'model': 0}, {'stick': {'colorscheme': 'Jmol', 'radius': 0.3}})
view.zoomTo()

def create_linked_control(desc, val, min_v, max_v, step, color=None):
    num = widgets.BoundedFloatText(value=val, min=min_v, max=max_v, step=step, description=desc, style={'description_width': 'initial'}, layout=widgets.Layout(width='160px'))
    slider = widgets.FloatSlider(value=val, min=min_v, max=max_v, step=step, readout=False, style={'handle_color': color} if color else {}, layout=widgets.Layout(width='100%'))
    widgets.link((num, 'value'), (slider, 'value'))
    return slider, widgets.VBox([num, slider], layout=widgets.Layout(align_items='center', padding='5px 15px'))

cx_sl, cx_ui = create_linked_control('中心 X:', vina_box['cx'], initial_center['cx']-padding, initial_center['cx']+padding, 0.5, 'red')
cy_sl, cy_ui = create_linked_control('中心 Y:', vina_box['cy'], initial_center['cy']-padding, initial_center['cy']+padding, 0.5, 'green')
cz_sl, cz_ui = create_linked_control('中心 Z:', vina_box['cz'], initial_center['cz']-padding, initial_center['cz']+padding, 0.5, 'blue')

sx_sl, sx_ui = create_linked_control('盒子 X:', vina_box['sx'], 5, 60, 1, 'red')
sy_sl, sy_ui = create_linked_control('盒子 Y:', vina_box['sy'], 5, 60, 1, 'green')
sz_sl, sz_ui = create_linked_control('盒子 Z:', vina_box['sz'], 5, 60, 1, 'blue')

btn_reset = widgets.Button(description="重置", button_style='info')
btn_confirm = widgets.Button(description="確認", button_style='success')
out_text, out_viewer = widgets.Output(), widgets.Output()

style_dropdown = widgets.Dropdown(options=[('Cartoon', 'Cartoon'), ('Licorice', 'Licorice'), ('Surface', 'Surface'), ('CPK', 'CPK')], value='Licorice', description='呈現方式:')
color_dropdown = widgets.Dropdown(options=[('Secondary Struct', 'ssJmol'), ('Atom Element', 'Jmol'), ('By Chain', 'chain')], value='Jmol', description='著色:')

def update_protein_style(*args):
    rep, color = style_dropdown.value, color_dropdown.value
    view.removeAllSurfaces()
    style_dict = {}
    if rep == 'Cartoon': style_dict = {'cartoon': {'colorscheme': color}}
    elif rep == 'Licorice': style_dict = {'stick': {'colorscheme': color, 'radius': 0.3}}
    elif rep == 'Surface': view.addSurface(py3Dmol.VDW, {'colorscheme': color}, {'model': 0})
    elif rep == 'CPK': style_dict = {'sphere': {'scale': 0.3, 'colorscheme': color}, 'stick': {'radius': 0.15, 'colorscheme': color}}
    view.setStyle({'model': 0}, style_dict)
    view.update()

style_dropdown.observe(update_protein_style, names='value')
color_dropdown.observe(update_protein_style, names='value')

last_draw_time = 0
def draw_box_shapes():
    view.removeAllShapes()
    c, dim = {'x': cx_sl.value, 'y': cy_sl.value, 'z': cz_sl.value}, {'w': sx_sl.value, 'h': sy_sl.value, 'd': sz_sl.value}
    hx, hy, hz = dim['w']/2, dim['h']/2, dim['d']/2

    # ★ 核心改寫：將單一實體 Box 拆解成 6 個極薄的彩色半透明平面 ★
    t = 0.05      # 面的厚度
    alpha = 0.8   # 半透明度

    # 1. YZ 平面 (左右兩面) - 對應 X 軸 (紅色)
    view.addBox({'center': {'x': c['x'] + hx, 'y': c['y'], 'z': c['z']}, 'dimensions': {'w': t, 'h': dim['h'], 'd': dim['d']}, 'color': 'red', 'opacity': alpha})
    view.addBox({'center': {'x': c['x'] - hx, 'y': c['y'], 'z': c['z']}, 'dimensions': {'w': t, 'h': dim['h'], 'd': dim['d']}, 'color': 'red', 'opacity': alpha})

    # 2. XZ 平面 (上下兩面) - 對應 Y 軸 (綠色)
    view.addBox({'center': {'x': c['x'], 'y': c['y'] + hy, 'z': c['z']}, 'dimensions': {'w': dim['w'], 'h': t, 'd': dim['d']}, 'color': 'green', 'opacity': alpha})
    view.addBox({'center': {'x': c['x'], 'y': c['y'] - hy, 'z': c['z']}, 'dimensions': {'w': dim['w'], 'h': t, 'd': dim['d']}, 'color': 'green', 'opacity': alpha})

    # 3. XY 平面 (前後兩面) - 對應 Z 軸 (藍色)
    view.addBox({'center': {'x': c['x'], 'y': c['y'], 'z': c['z'] + hz}, 'dimensions': {'w': dim['w'], 'h': dim['h'], 'd': t}, 'color': 'blue', 'opacity': alpha})
    view.addBox({'center': {'x': c['x'], 'y': c['y'], 'z': c['z'] - hz}, 'dimensions': {'w': dim['w'], 'h': dim['h'], 'd': t}, 'color': 'blue', 'opacity': alpha})

    # 繪製 12 條邊框線 (保持清晰立體感)
    corners = [(c['x']+i*hx, c['y']+j*hy, c['z']+k*hz) for i in [-1,1] for j in [-1,1] for k in [-1,1]]
    edges = [(0,1), (2,3), (4,5), (6,7), (0,2), (1,3), (4,6), (5,7), (0,4), (1,5), (2,6), (3,7)]
    for p1, p2 in edges:
        view.addCylinder({'start': {'x': corners[p1][0], 'y': corners[p1][1], 'z': corners[p1][2]}, 'end': {'x': corners[p2][0], 'y': corners[p2][1], 'z': corners[p2][2]}, 'radius': 0.1, 'color': 'darkgrey'})


def handle_slider_change(change):
    global last_draw_time
    if time.time() - last_draw_time < 0.05: return
    last_draw_time = time.time()
    draw_box_shapes()
    view.update()

for s in [cx_sl, cy_sl, cz_sl, sx_sl, sy_sl, sz_sl]: s.observe(handle_slider_change, names='value')
draw_box_shapes()

def reset_to_center(b):
    cx_sl.value, cy_sl.value, cz_sl.value = initial_center['cx'], initial_center['cy'], initial_center['cz']
btn_reset.on_click(reset_to_center)

def lock_params(b):
    global vina_box
    vina_box = {'cx': round(cx_sl.value, 1), 'cy': round(cy_sl.value, 1), 'cz': round(cz_sl.value, 1), 'sx': int(round(sx_sl.value)), 'sy': int(round(sy_sl.value)), 'sz': int(round(sz_sl.value))}
    controls_ui.layout.display = 'none'
    with out_text:
        clear_output(wait=True)
        print(f"[鎖定] Grid Box: 中心({vina_box['cx']}, {vina_box['cy']}, {vina_box['cz']}) | 尺寸({vina_box['sx']}, {vina_box['sy']}, {vina_box['sz']})")
btn_confirm.on_click(lock_params)

controls_ui = widgets.VBox([
    widgets.HBox([style_dropdown, color_dropdown], layout=widgets.Layout(justify_content='center', margin='10px 0')),
    widgets.HBox([cx_ui, cy_ui, cz_ui], layout=widgets.Layout(justify_content='center')),
    widgets.HBox([sx_ui, sy_ui, sz_ui], layout=widgets.Layout(justify_content='center')),
    widgets.HBox([btn_reset, btn_confirm], layout=widgets.Layout(justify_content='center', margin='10px 0'))
])

display(widgets.VBox([controls_ui, out_text, out_viewer]))
with out_viewer:
    clear_output(wait=True)
    view.show()

[資訊] 蛋白質跨度: 47.27 Å | 動態邊界鎖定: 質心 +/- 53.64 Å


### Vina 核心演算與進度監控
功能說明：掛載 CPU 版本的 Vina 引擎進行剛性與柔性對接運算。

核心設計：強制啟用 exhaustiveness (窮舉度) 的最高演算設定，最大化搜索空間內所有潛在的構象 (Conformations)。

In [ ]:
class Logger(object):
    def __init__(self):
        self.terminal = sys.stdout
        self.log = open("vina_docking_log.txt", "w")
    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)
        self.flush()
    def flush(self):
        self.terminal.flush()
        self.log.flush()

sys.stdout = Logger()
v = Vina(sf_name='vina')
v.set_receptor('receptor.pdbqt')
v.set_ligand_from_file('ligand.pdbqt')
v.compute_vina_maps(center=[vina_box['cx'], vina_box['cy'], vina_box['cz']], box_size=[vina_box['sx'], vina_box['sy'], vina_box['sz']])

print(f"[執行] 開始 Docking... (中心: {vina_box['cx']},{vina_box['cy']},{vina_box['cz']} | 尺寸: {vina_box['sx']}x{vina_box['sy']}x{vina_box['sz']})")
v.dock(exhaustiveness=8, n_poses=9)
v.write_poses('output.pdbqt', n_poses=9, overwrite=True)

sys.stdout = sys.__stdout__
print("\n[完成] Docking 運算結束，結果已保存。")

def summarize_vina_results(input_file, summary_file):
    if not os.path.exists(input_file): return print("[錯誤] 找不到 Docking 結果檔案。")
    results = []
    regex = re.compile(r'^REMARK VINA RESULT:\s+(-?\d+\.\d+)\s+(\d+\.\d+)\s+(\d+\.\d+)')

    with open(input_file, 'r') as f:
        for line in f:
            match = regex.match(line)
            if match: results.append(match.groups())

    with open(summary_file, 'w', encoding='utf-8') as out:
        out.write("=== AutoDock Vina 結果摘要 ===\n")
        out.write(f"{'Mode':<6} | {'Affinity (kcal/mol)':<20} | {'RMSD l.b.':<10} | {'RMSD u.b.':<10}\n")
        out.write("-" * 55 + "\n")
        for i, r in enumerate(results):
            out.write(f"Pose {i+1:<1} | {r[0]:<20} | {r[1]:<10} | {r[2]:<10}\n")

    print(f"\n[報告] 摘要已匯出至 {summary_file}")
    with open(summary_file, 'r', encoding='utf-8') as f: print(f.read())

summarize_vina_results('output.pdbqt', 'docking_summary_report.txt')

[執行] 開始 Docking... (中心: 15.0,0.0,20.0 | 尺寸: 20x20x20)


### 結合能分析與 3D 視覺化呈現
功能說明：解析 PDBQT 輸出檔，提取所有預測構象 (Poses) 及其熱力學親和力 (Affinity, kcal/mol)。

核心設計：整合 py3Dmol 構建互動式結果檢視面板。支援無縫切換不同對接姿態，並能即時渲染配體與周圍氨基酸的空間位阻 (Steric Clash) ，為結構修飾 (Structure-based Drug Design) 提供直覺的決策輔助。

In [ ]:
print("[處理] 正在準備 3D 視覺化結果...")

pose_scores = []
pose_files = []

# 直接解析 Vina 原始的 output.pdbqt，手動切分後命名為 output_pose_n.pdb
if os.path.exists('output.pdbqt'):
    with open('output.pdbqt', 'r') as f:
        lines = f.readlines()

    # 抓取親和力分數
    pose_scores = [float(line.split()[3]) for line in lines if line.startswith("REMARK VINA RESULT")]

    # 手動切分構象，然後再一對一轉成 PDB
    model_lines = []
    pose_idx = 1
    for line in lines:
        model_lines.append(line)
        if line.startswith("ENDMDL"):
            # 先存成獨立的暫存 pdbqt
            tmp_pdbqt = f"temp_pose_{pose_idx}.pdbqt"
            with open(tmp_pdbqt, 'w') as out_f:
                out_f.writelines(model_lines)

            # 呼叫 subprocess 轉成 PDB，採用新的輸出命名格式
            out_pdb = f"output_pose_{pose_idx}.pdb"
            subprocess.run(
                f'obabel -ipdbqt {tmp_pdbqt} -opdb -O {out_pdb}',
                shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
            )

            # 記錄成功產生的檔案並刪除暫存
            if os.path.exists(out_pdb):
                pose_files.append(out_pdb)
            os.remove(tmp_pdbqt)

            pose_idx += 1
            model_lines = []

carbon_schemes = ['magentaCarbon', 'cyanCarbon', 'yellowCarbon', 'greenCarbon', 'orangeCarbon', 'purpleCarbon', 'aliceblueCarbon', 'pinkCarbon', 'grayCarbon']

view = py3Dmol.view(width=800, height=600)
# 保留微淺灰背景，讓整體視覺更清晰舒適
view.setBackgroundColor('#f8f9fa')

# 智慧讀取受體 (優先讀取系統標準化的 pdb，若無則讀取 pdbqt)
rec_target = 'receptor_original.pdb' if os.path.exists('receptor_original.pdb') else 'receptor.pdbqt'
if os.path.exists(rec_target):
    with open(rec_target, 'r') as f:
        view.addModel(f.read(), 'pdb' if rec_target.endswith('.pdb') else 'pdbqt')

view.setStyle({'model': 0}, {'stick': {'colorscheme': 'Jmol', 'radius': 0.3, 'opacity': 1.0}})

# 載入所有切好的配體姿態 (output_pose_n.pdb)
for p_file in pose_files:
    with open(p_file, 'r') as f: view.addModel(f.read(), 'pdb')

view.zoomTo({'model': 0})
n_poses = len(pose_scores)

for i in range(1, n_poses + 1):
    view.setStyle({'model': i}, {'stick': {'colorscheme': carbon_schemes[0], 'radius': 0.35}} if i == 1 else {})

# UI 控制項 (移除氫原子切換鈕)
style_dropdown_prot = widgets.Dropdown(options=[('Cartoon', 'Cartoon'), ('Licorice', 'Licorice'), ('Surface', 'Surface'), ('CPK', 'CPK')], value='Licorice', description='蛋白質呈現:')
color_dropdown_prot = widgets.Dropdown(options=[('Secondary Struct', 'ssJmol'), ('Atom Element', 'Jmol'), ('By Chain', 'chain')], value='Jmol', description='著色:')
opacity_checkbox_prot = widgets.Checkbox(value=False, description='半透明模式', indent=False)

def update_render(*args):
    alpha = 0.5 if opacity_checkbox_prot.value else 1.0
    view.removeAllSurfaces()
    style_dict = {}

    if style_dropdown_prot.value == 'Cartoon': style_dict = {'cartoon': {'colorscheme': color_dropdown_prot.value, 'opacity': alpha}}
    elif style_dropdown_prot.value == 'Licorice': style_dict = {'stick': {'colorscheme': color_dropdown_prot.value, 'radius': 0.3, 'opacity': alpha}}
    elif style_dropdown_prot.value == 'Surface': view.addSurface(py3Dmol.VDW, {'opacity': alpha, 'colorscheme': color_dropdown_prot.value}, {'model': 0})
    elif style_dropdown_prot.value == 'CPK': style_dict = {'sphere': {'scale': 0.3, 'colorscheme': color_dropdown_prot.value, 'opacity': alpha}, 'stick': {'radius': 0.15, 'colorscheme': color_dropdown_prot.value, 'opacity': alpha}}

    view.setStyle({'model': 0}, style_dict)

    sel = dropdown_pose.value
    for i in range(1, len(pose_files) + 1):
        if sel == "ALL" or sel == i - 1:
            color = carbon_schemes[(i-1) % len(carbon_schemes)]
            # 拿掉隱藏氫原子的判斷，完整呈現 PDB 內容
            view.setStyle({'model': i}, {'stick': {'colorscheme': color, 'radius': 0.3}})
        else:
            view.setStyle({'model': i}, {})

    view.update()

style_dropdown_prot.observe(update_render, names='value')
color_dropdown_prot.observe(update_render, names='value')
opacity_checkbox_prot.observe(update_render, names='value')

opts = [(f"Pose {i+1} ({pose_scores[i]} kcal/mol)", i) for i in range(len(pose_files))] + [("🌈 顯示所有姿態 (Compare All)", "ALL")]
dropdown_pose = widgets.Dropdown(options=opts, value=0, description='🎯 選擇姿態:', layout=widgets.Layout(width='400px'))

out_text, out_viewer = widgets.Output(), widgets.Output()

def update_pose_view(*args):
    sel = dropdown_pose.value
    with out_text:
        clear_output(wait=True)
        print(f"[狀態] 模式：全姿態對比" if sel == "ALL" else f"[狀態] 模式：單一姿態 | Pose {sel + 1} | 親和力: {pose_scores[sel]} kcal/mol")
    update_render()

dropdown_pose.observe(update_pose_view, names='value')

with out_text:
    if len(pose_files) > 0:
        print(f"[狀態] 模式：單一姿態 | Pose 1 | 親和力: {pose_scores[0]} kcal/mol")
    else:
        print("[錯誤] 找不到任何對接結果，請檢查 Vina 運算是否成功。")

with out_viewer:
    clear_output(wait=True)
    view.show()

display(widgets.VBox([
    widgets.HBox([style_dropdown_prot, color_dropdown_prot, opacity_checkbox_prot], layout=widgets.Layout(justify_content='center', margin='10px 0')),
    widgets.HBox([dropdown_pose], layout=widgets.Layout(justify_content='center', margin='5px 0')),
    out_text, out_viewer
]))

In [ ]:
# 產生帶時間戳記的檔名
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
zip_filename = f"docking_results_{timestamp}.zip"

# 定義要抓取的「標準化關鍵字」前綴
include_prefixes = (
    'receptor',        # 包含 receptor_original.*, receptor.pdbqt
    'ligand',          # 包含 ligand_original.*, ligand.mol2, ligand.pdb, ligand.pdbqt
    'output',          # 包含所有對接構象 output_pose_1.pdb 等
    'docking_summary', # 包含 docking_summary_report.txt
    'vina_docking'     # 包含 vina_docking_log.txt
)

print(f"[打包] 正在精準選取實驗數據並壓縮至 {zip_filename} ...")

file_count = 0

with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files_in_dir in os.walk('.'):
        # 排除 Colab 的系統資料夾
        dirs[:] = [d for d in dirs if d not in ['sample_data', '.config']]

        for file in files_in_dir:
            # 判斷邏輯：檔名開頭符合我們的關鍵字，且排除壓縮檔本身
            if file.startswith(include_prefixes) and file != zip_filename:
                file_path = os.path.join(root, file)
                # arcname=file 確保解壓縮後檔案不會帶有 /content 等路徑結構
                zipf.write(file_path, arcname=file)
                print(f"  + 匯入: {file}")
                file_count += 1

if file_count > 0:
    print(f"\n[完成] 共成功打包 {file_count} 個檔案！自動啟動下載...")
    files.download(zip_filename)
else:
    print("\n[警告] 找不到任何符合關鍵字的實驗檔案，請確認先前的步驟是否執行成功。")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>